# DAM2 Sleep Analysis — reusable workflow

Workflow
1. Configuration
2. Import and preparation of DAM2 data
3. Total sleep time
4. Total sleep time by Light/Dark phase
5. Sleep-bout duration and number
6. Sleep latency after lights-off
7. Hourly sleep profile (before vs after)
8. Combined export for statistical analysis in R

Important:
- Sleep is defined here as one 5-min bin with activity == 0.
- "Before" = first 24 h, "after" = second 24 h.
- Light/Dark is derived exclusively from the LD column (1 = Light, 0 = Dark).
- Outlier cutoffs are used only for visualization. The R export contains the
  complete, unfiltered values unless values are genuinely missing.

For a new fragrance, normally only edit the CONFIGURATION section below.

In [ ]:
# ============================================================
# 1) IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from IPython.display import display


# ------------------------------------------------------------
# EDIT THESE SETTINGS FOR EACH NEW FRAGRANCE
# ------------------------------------------------------------

FRAGRANCE_NAME = "F1"
CONCENTRATION_LABEL = "40µL"

# The filenames are generated automatically from the settings above.
# Change them here only if your files use a different naming convention.
REPLICATES = [
    {
        "replicate": "R1",
        "filepath": f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}_Rep_1.csv",
        "dead_flies": None,
    },
    {
        "replicate": "R2",
        "filepath": f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}_Rep_2.csv",
        "dead_flies": [5, 6, 30],
    },
]

# Output folder and file prefix
OUTPUT_DIR = Path(f"sleep_analysis_{FRAGRANCE_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREFIX = f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}".replace("µ", "u")

# DAM2 settings
BIN_LENGTH_MIN = 5
BINS_PER_HOUR = 12
MAX_HOURS = 48

# Plot-only cutoffs (do NOT affect R export)
SLEEP_LATENCY_PLOT_CUTOFF_MIN = 80
BOUT_DURATION_PLOT_CUTOFF_MIN = 150

# Display order
CONDITION_ORDER = ["before", "after"]
PHASE_ORDER = ["Light", "Dark"]

# Plot colors
condition_palette = {
    "before": "#fff7b2",
    "after": "#9ccfc8",
}
point_palette = {
    "before": "#9a8f35",
    "after": "#3f8f8b",
}

plt.style.use("default")
sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
})

In [ ]:
# ============================================================
# 2) HELPER FUNCTIONS
# ============================================================

def add_phase_from_ld(data, ld_col="LD"):
    """Add categorical Light/Dark phase from DAM LD coding."""
    data = data.copy()
    data[ld_col] = pd.to_numeric(data[ld_col], errors="coerce")

    phase = pd.Series(pd.NA, index=data.index, dtype="object")
    phase.loc[data[ld_col] == 1] = "Light"
    phase.loc[data[ld_col] == 0] = "Dark"

    data["Phase"] = pd.Categorical(
        phase,
        categories=PHASE_ORDER,
        ordered=True,
    )
    return data


def load_dam_sleep_csv(filepath, replicate, dead_flies=None):
    """
    Load one DAM2 CSV and return long-format 5-min data.

    Expected DAM structure after removing unused columns:
    Index, Date, Time, LD, Fliege1 ... Fliege32
    """
    data = pd.read_csv(filepath, sep=";", header=None)

    # Unused DAM export columns
    data = data.drop(columns=[3, 4, 5, 6, 7, 8], errors="ignore")

    expected_cols = 4 + 32
    if data.shape[1] != expected_cols:
        raise ValueError(
            f"{filepath}: found {data.shape[1]} columns after cleaning; "
            f"expected {expected_cols}. Check the DAM export format."
        )

    data.columns = ["Index", "Date", "Time", "LD"] + [
        f"Fliege{i}" for i in range(1, 33)
    ]

    if dead_flies:
        dead_columns = [f"Fliege{i}" for i in dead_flies]
        data = data.drop(columns=dead_columns, errors="ignore")

    data["Index"] = pd.to_numeric(data["Index"], errors="coerce")
    data["LD"] = pd.to_numeric(data["LD"], errors="coerce")

    data["Replicate"] = replicate
    data["Hour_raw"] = (data["Index"] - 1) // BINS_PER_HOUR
    data = data[data["Hour_raw"] < MAX_HOURS].copy()

    fly_columns = [col for col in data.columns if col.startswith("Fliege")]

    long = data.melt(
        id_vars=["Replicate", "Index", "Hour_raw", "Date", "Time", "LD"],
        value_vars=fly_columns,
        var_name="Fliege",
        value_name="activity",
    )

    long["activity"] = pd.to_numeric(long["activity"], errors="coerce")

    long["ID"] = (
        long["Replicate"].astype(str)
        + "_"
        + long["Fliege"].astype(str)
    )

    long["condition"] = np.where(
        long["Hour_raw"] < 24,
        "before",
        "after",
    )
    long["condition"] = pd.Categorical(
        long["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    long["Hour"] = long["Hour_raw"] % 24

    # 5 min with no beam crossing = sleep bin
    long["sleep_bin"] = long["activity"] == 0

    long = add_phase_from_ld(long, ld_col="LD")
    return long


def save_figure(fig, filename):
    """Save plot with common settings."""
    path = OUTPUT_DIR / filename
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    print(f"Saved plot: {path}")


def add_mean_median_legend(ax):
    handles = [
        Line2D([0], [0], color="black", linestyle="--",
               linewidth=1.2, label="Mean"),
        Line2D([0], [0], color="black", linestyle="-",
               linewidth=1.0, label="Median"),
    ]
    ax.legend(
        handles=handles,
        frameon=True,
        facecolor="white",
        edgecolor="black",
    )


def plot_condition_box(data, y, ylabel, title, filename):
    """Reusable before-vs-after box + swarm plot."""
    fig, ax = plt.subplots(figsize=(7.5, 5))

    sns.boxplot(
        data=data,
        x="condition",
        y=y,
        order=CONDITION_ORDER,
        hue="condition",
        hue_order=CONDITION_ORDER,
        palette=condition_palette,
        dodge=False,
        showmeans=True,
        meanline=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.0},
        meanprops={"color": "black", "linewidth": 1.2, "linestyle": "--"},
        ax=ax,
    )

    sns.swarmplot(
        data=data,
        x="condition",
        y=y,
        order=CONDITION_ORDER,
        hue="condition",
        hue_order=CONDITION_ORDER,
        palette=point_palette,
        dodge=False,
        size=5,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
    )

    if ax.legend_ is not None:
        ax.legend_.remove()
    add_mean_median_legend(ax)

    ax.set_title(f"{FRAGRANCE_NAME}: {title}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Condition")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()


def plot_phase_box(data, y, ylabel, title, filename):
    """Reusable Light/Dark × before/after box + swarm plot."""
    fig, ax = plt.subplots(figsize=(9, 5))

    sns.boxplot(
        data=data,
        x="Phase",
        y=y,
        hue="condition",
        order=PHASE_ORDER,
        hue_order=CONDITION_ORDER,
        palette=condition_palette,
        showmeans=True,
        meanline=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.0},
        meanprops={"color": "black", "linewidth": 1.2, "linestyle": "--"},
        ax=ax,
    )

    sns.swarmplot(
        data=data,
        x="Phase",
        y=y,
        hue="condition",
        order=PHASE_ORDER,
        hue_order=CONDITION_ORDER,
        palette=point_palette,
        dodge=True,
        size=5,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
    )

    # Keep only one set of condition labels
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    condition_handles = [unique[c] for c in CONDITION_ORDER if c in unique]
    condition_labels = [c for c in CONDITION_ORDER if c in unique]

    mean_median_handles = [
        Line2D([0], [0], color="black", linestyle="--",
               linewidth=1.2, label="Mean"),
        Line2D([0], [0], color="black", linestyle="-",
               linewidth=1.0, label="Median"),
    ]

    ax.legend(
        condition_handles + mean_median_handles,
        condition_labels + ["Mean", "Median"],
        title="Condition",
        frameon=True,
        facecolor="white",
        edgecolor="black",
    )

    ax.set_title(f"{FRAGRANCE_NAME}: {title}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Phase")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()


def calculate_bouts(data, split_at_phase=False):
    """
    Return individual sleep bouts.

    split_at_phase=False:
        a bout is defined across the full 24-h condition.

    split_at_phase=True:
        a bout is additionally split at Light/Dark transitions so that
        phase-specific bout metrics can be calculated unambiguously.
    """
    bout_data = data.sort_values(
        ["ID", "condition", "Index"]
    ).reset_index(drop=True).copy()

    group_cols = ["ID", "condition"]

    bout_data["prev_sleep_bin"] = (
        bout_data.groupby(group_cols, observed=True)["sleep_bin"].shift()
    )

    new_segment = (
        bout_data["prev_sleep_bin"].isna()
        | (bout_data["sleep_bin"] != bout_data["prev_sleep_bin"])
    )

    if split_at_phase:
        bout_data["prev_phase"] = (
            bout_data.groupby(group_cols, observed=True)["Phase"].shift()
        )
        new_segment = new_segment | (
            bout_data["Phase"] != bout_data["prev_phase"]
        )

    bout_data["new_segment"] = new_segment
    bout_data["segment_id"] = (
        bout_data.groupby(group_cols, observed=True)["new_segment"].cumsum()
    )

    grouping = ["ID", "Replicate", "condition"]
    if split_at_phase:
        grouping.append("Phase")
    grouping.append("segment_id")

    bouts = (
        bout_data[bout_data["sleep_bin"]]
        .groupby(grouping, observed=True)
        .agg(
            n_bins=("sleep_bin", "size"),
            start_index=("Index", "min"),
            end_index=("Index", "max"),
        )
        .reset_index()
    )

    bouts["sleep_bout_duration_min"] = bouts["n_bins"] * BIN_LENGTH_MIN
    return bouts


def calculate_sleep_latency_first_sleep_bin(data):
    """
    Sleep latency = time from LD transition 1 -> 0 until the first sleep bin.
    If the first dark bin is already sleep, latency = 0 min.
    """
    records = []

    latency_data = data.copy()
    latency_data["LD"] = pd.to_numeric(latency_data["LD"], errors="coerce")
    latency_data["Index"] = pd.to_numeric(
        latency_data["Index"], errors="coerce"
    )

    for (fly_id, condition), group in latency_data.groupby(
        ["ID", "condition"], observed=True
    ):
        group = group.sort_values("Index").reset_index(drop=True)

        prev_ld = group["LD"].shift()
        lights_off_positions = group.index[
            (prev_ld == 1) & (group["LD"] == 0)
        ].tolist()

        # Under a normal 12:12 LD schedule there should be one lights-off
        # transition per 24-h condition.
        for pos in lights_off_positions:
            lights_off_row = group.loc[pos]

            after_start = group.loc[pos:].reset_index(drop=True)
            next_light_positions = after_start.index[
                (after_start.index > 0) & (after_start["LD"] == 1)
            ].tolist()

            if next_light_positions:
                dark_block = after_start.loc[
                    : next_light_positions[0] - 1
                ].copy()
            else:
                dark_block = after_start.copy()

            first_sleep = dark_block.loc[dark_block["sleep_bin"]].head(1)

            if first_sleep.empty:
                first_sleep_index = np.nan
                first_sleep_time = np.nan
                latency_min = np.nan
            else:
                first_sleep_row = first_sleep.iloc[0]
                first_sleep_index = first_sleep_row["Index"]
                first_sleep_time = first_sleep_row["Time"]
                latency_min = (
                    first_sleep_index - lights_off_row["Index"]
                ) * BIN_LENGTH_MIN

            records.append({
                "ID": fly_id,
                "Replicate": lights_off_row["Replicate"],
                "Fliege": lights_off_row["Fliege"],
                "condition": condition,
                "lights_off_index": lights_off_row["Index"],
                "lights_off_time": lights_off_row["Time"],
                "first_sleep_index": first_sleep_index,
                "first_sleep_time": first_sleep_time,
                "sleep_latency_min": latency_min,
            })

    result = pd.DataFrame(records)

    if not result.empty:
        result["condition"] = pd.Categorical(
            result["condition"],
            categories=CONDITION_ORDER,
            ordered=True,
        )

    return result

In [ ]:
# ============================================================
# 3) LOAD AND PREPARE DATA
# ============================================================

replicate_frames = []

for config in REPLICATES:
    frame = load_dam_sleep_csv(
        filepath=config["filepath"],
        replicate=config["replicate"],
        dead_flies=config["dead_flies"],
    )
    replicate_frames.append(frame)

df_sleep = pd.concat(replicate_frames, ignore_index=True)

df_sleep = (
    df_sleep
    .sort_values(["Replicate", "Fliege", "Index"])
    .reset_index(drop=True)
)

df_sleep["Fragrance"] = FRAGRANCE_NAME
df_sleep["Concentration"] = CONCENTRATION_LABEL

print(f"{FRAGRANCE_NAME}: raw long-format sleep data")
print("Shape:", df_sleep.shape)
print("Number of flies:", df_sleep["ID"].nunique())
display(df_sleep.head())

# Basic checks
print("\nObservations per fly:")
print(df_sleep.groupby("ID").size().value_counts().sort_index())

print("\nMissing activity values:", df_sleep["activity"].isna().sum())
print("Missing LD values:", df_sleep["LD"].isna().sum())

In [ ]:
# ============================================================
# 4) TOTAL SLEEP TIME — BEFORE VS AFTER
# ============================================================

df_total_sleep = (
    df_sleep
    .groupby(["ID", "Replicate", "condition"], observed=True)["sleep_bin"]
    .sum()
    .reset_index(name="n_sleep_bins")
)

df_total_sleep["total_sleep_min"] = (
    df_total_sleep["n_sleep_bins"] * BIN_LENGTH_MIN
)

display(df_total_sleep.head())

plot_condition_box(
    data=df_total_sleep,
    y="total_sleep_min",
    ylabel="Total sleep time [min]",
    title="Total Sleep Time",
    filename=f"{OUTPUT_PREFIX}_total_sleep.png",
)

In [ ]:
# ============================================================
# 5) TOTAL SLEEP TIME BY LIGHT/DARK
# ============================================================

df_total_sleep_phase = (
    df_sleep
    .dropna(subset=["Phase"])
    .groupby(
        ["ID", "Replicate", "condition", "Phase"],
        observed=True,
    )["sleep_bin"]
    .sum()
    .reset_index(name="n_sleep_bins")
)

df_total_sleep_phase["total_sleep_min"] = (
    df_total_sleep_phase["n_sleep_bins"] * BIN_LENGTH_MIN
)

display(df_total_sleep_phase.head())

plot_phase_box(
    data=df_total_sleep_phase,
    y="total_sleep_min",
    ylabel="Total sleep time [min]",
    title="Total Sleep Time by Light/Dark Phase",
    filename=f"{OUTPUT_PREFIX}_total_sleep_by_phase.png",
)

In [ ]:
# ============================================================
# 6) SLEEP BOUTS — OVERALL 24-H CONDITION
# ============================================================

sleep_bouts_overall = calculate_bouts(
    df_sleep,
    split_at_phase=False,
)

# Complete base ensures flies with zero bouts are retained.
base_overall = (
    df_sleep[["ID", "Replicate", "condition"]]
    .drop_duplicates()
)

mean_bout_overall = (
    sleep_bouts_overall
    .groupby(
        ["ID", "Replicate", "condition"],
        observed=True,
    )["sleep_bout_duration_min"]
    .mean()
    .reset_index(name="mean_sleep_bout_duration_min")
)

bout_number_overall = (
    sleep_bouts_overall
    .groupby(
        ["ID", "Replicate", "condition"],
        observed=True,
    )
    .size()
    .reset_index(name="sleep_bout_number")
)

df_bout_overall = (
    base_overall
    .merge(
        mean_bout_overall,
        on=["ID", "Replicate", "condition"],
        how="left",
    )
    .merge(
        bout_number_overall,
        on=["ID", "Replicate", "condition"],
        how="left",
    )
)

df_bout_overall["sleep_bout_number"] = (
    df_bout_overall["sleep_bout_number"]
    .fillna(0)
    .astype(int)
)

# Mean bout duration is undefined when a fly has zero bouts, so NaN is retained.

display(df_bout_overall.head())

plot_condition_box(
    data=df_bout_overall.dropna(
        subset=["mean_sleep_bout_duration_min"]
    ),
    y="mean_sleep_bout_duration_min",
    ylabel="Mean sleep bout duration [min]",
    title="Mean Sleep Bout Duration",
    filename=f"{OUTPUT_PREFIX}_mean_bout_duration.png",
)

plot_condition_box(
    data=df_bout_overall,
    y="sleep_bout_number",
    ylabel="Number of sleep bouts",
    title="Number of Sleep Bouts",
    filename=f"{OUTPUT_PREFIX}_sleep_bout_number.png",
)

In [ ]:
# ============================================================
# 7) SLEEP BOUTS BY LIGHT/DARK PHASE
# ============================================================

sleep_bouts_phase = calculate_bouts(
    df_sleep.dropna(subset=["Phase"]),
    split_at_phase=True,
)

base_phase = (
    df_sleep
    .dropna(subset=["Phase"])
    [["ID", "Replicate", "condition", "Phase"]]
    .drop_duplicates()
)

mean_bout_phase = (
    sleep_bouts_phase
    .groupby(
        ["ID", "Replicate", "condition", "Phase"],
        observed=True,
    )["sleep_bout_duration_min"]
    .mean()
    .reset_index(name="mean_sleep_bout_duration_min")
)

bout_number_phase = (
    sleep_bouts_phase
    .groupby(
        ["ID", "Replicate", "condition", "Phase"],
        observed=True,
    )
    .size()
    .reset_index(name="sleep_bout_number")
)

df_bout_phase = (
    base_phase
    .merge(
        mean_bout_phase,
        on=["ID", "Replicate", "condition", "Phase"],
        how="left",
    )
    .merge(
        bout_number_phase,
        on=["ID", "Replicate", "condition", "Phase"],
        how="left",
    )
)

df_bout_phase["sleep_bout_number"] = (
    df_bout_phase["sleep_bout_number"]
    .fillna(0)
    .astype(int)
)

display(df_bout_phase.head())

# Plot-only filtered version. Original data remain unchanged.
bout_plot_data = df_bout_phase[
    df_bout_phase["mean_sleep_bout_duration_min"]
    <= BOUT_DURATION_PLOT_CUTOFF_MIN
].copy()

n_bout_plot_excluded = (
    df_bout_phase["mean_sleep_bout_duration_min"]
    > BOUT_DURATION_PLOT_CUTOFF_MIN
).sum()

print(
    f"Mean-bout values > {BOUT_DURATION_PLOT_CUTOFF_MIN} min "
    f"not shown in phase plot: {n_bout_plot_excluded}"
)

plot_phase_box(
    data=bout_plot_data.dropna(
        subset=["mean_sleep_bout_duration_min"]
    ),
    y="mean_sleep_bout_duration_min",
    ylabel="Mean sleep bout duration [min]",
    title=(
        "Mean Sleep Bout Duration by Light/Dark Phase "
        f"(>{BOUT_DURATION_PLOT_CUTOFF_MIN} min not shown)"
    ),
    filename=f"{OUTPUT_PREFIX}_mean_bout_duration_by_phase.png",
)

plot_phase_box(
    data=df_bout_phase,
    y="sleep_bout_number",
    ylabel="Number of sleep bouts",
    title="Number of Sleep Bouts by Light/Dark Phase",
    filename=f"{OUTPUT_PREFIX}_sleep_bout_number_by_phase.png",
)

In [ ]:
# ============================================================
# 8) SLEEP LATENCY AFTER LIGHTS-OFF
# ============================================================

sleep_latency_df = calculate_sleep_latency_first_sleep_bin(df_sleep)

display(sleep_latency_df.head())

# Plot only: exclude > cutoff visually, but retain them in R export.
sleep_latency_plot_df = sleep_latency_df[
    sleep_latency_df["sleep_latency_min"]
    <= SLEEP_LATENCY_PLOT_CUTOFF_MIN
].copy()

n_latency_plot_excluded = (
    sleep_latency_df["sleep_latency_min"]
    > SLEEP_LATENCY_PLOT_CUTOFF_MIN
).sum()

print(
    f"Sleep-latency values > {SLEEP_LATENCY_PLOT_CUTOFF_MIN} min "
    f"not shown in plot: {n_latency_plot_excluded}"
)

plot_condition_box(
    data=sleep_latency_plot_df.dropna(
        subset=["sleep_latency_min"]
    ),
    y="sleep_latency_min",
    ylabel="Sleep latency [min]",
    title=(
        "Sleep Latency after Lights-Off "
        f"(>{SLEEP_LATENCY_PLOT_CUTOFF_MIN} min not shown)"
    ),
    filename=f"{OUTPUT_PREFIX}_sleep_latency.png",
)

In [ ]:
# ============================================================
# 9) HOURLY SLEEP PROFILE — BEFORE VS AFTER
# ============================================================

# Per fly:
# sleep_min_hour = number of sleeping 5-min bins within each hour * 5 min
df_hourly_sleep = (
    df_sleep
    .groupby(
        ["ID", "Replicate", "condition", "Hour"],
        observed=True,
    )
    .agg(
        n_sleep_bins=("sleep_bin", "sum"),
        n_valid_bins=("sleep_bin", "count"),
    )
    .reset_index()
)

df_hourly_sleep["sleep_min_hour"] = (
    df_hourly_sleep["n_sleep_bins"] * BIN_LENGTH_MIN
)

df_hourly_sleep["sleep_fraction"] = (
    df_hourly_sleep["n_sleep_bins"]
    / df_hourly_sleep["n_valid_bins"]
)

# Mean across individual flies for every condition × hour
hourly_sleep_summary = (
    df_hourly_sleep
    .groupby(["condition", "Hour"], observed=True)
    .agg(
        n=("sleep_min_hour", "count"),
        mean_sleep_min=("sleep_min_hour", "mean"),
        sd_sleep_min=("sleep_min_hour", "std"),
    )
    .reset_index()
)

hourly_sleep_summary["se_sleep_min"] = (
    hourly_sleep_summary["sd_sleep_min"]
    / np.sqrt(hourly_sleep_summary["n"])
)

fig, ax = plt.subplots(figsize=(10, 5.5))

for condition in CONDITION_ORDER:
    condition_data = (
        hourly_sleep_summary[
            hourly_sleep_summary["condition"] == condition
        ]
        .sort_values("Hour")
    )

    x = condition_data["Hour"].to_numpy(dtype=float)
    y = condition_data["mean_sleep_min"].to_numpy(dtype=float)
    se = condition_data["se_sleep_min"].to_numpy(dtype=float)

    ax.plot(
        x,
        y,
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=condition,
        color=point_palette[condition],
    )

    ax.fill_between(
        x,
        y - se,
        y + se,
        alpha=0.18,
        color=point_palette[condition],
    )

ax.set_title(
    f"{FRAGRANCE_NAME}: Hourly Sleep Profile",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Hour of 24-h cycle")
ax.set_ylabel("Mean sleep per hour [min]")
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(0, 23)
ax.set_ylim(0, 60)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(title="Condition")

plt.tight_layout()
save_figure(
    fig,
    f"{OUTPUT_PREFIX}_hourly_sleep_profile.png",
)
plt.show()

In [ ]:
# ============================================================
# 10) BUILD ONE COMBINED DATAFRAME FOR R
# ============================================================

# One row = one fly × condition.
# This is convenient for separate R models of each sleep endpoint.

df_sleep_parameters_R = (
    df_total_sleep[
        ["ID", "Replicate", "condition", "total_sleep_min"]
    ]
    .merge(
        df_bout_overall[
            [
                "ID",
                "Replicate",
                "condition",
                "mean_sleep_bout_duration_min",
                "sleep_bout_number",
            ]
        ],
        on=["ID", "Replicate", "condition"],
        how="left",
    )
    .merge(
        sleep_latency_df[
            [
                "ID",
                "Replicate",
                "condition",
                "sleep_latency_min",
            ]
        ],
        on=["ID", "Replicate", "condition"],
        how="left",
    )
)

# ------------------------------------------------------------
# Add Light/Dark total sleep as separate columns
# ------------------------------------------------------------

phase_sleep_wide = (
    df_total_sleep_phase
    .pivot(
        index=["ID", "Replicate", "condition"],
        columns="Phase",
        values="total_sleep_min",
    )
    .rename(
        columns={
            "Light": "total_sleep_light_min",
            "Dark": "total_sleep_dark_min",
        }
    )
    .reset_index()
)

df_sleep_parameters_R = df_sleep_parameters_R.merge(
    phase_sleep_wide,
    on=["ID", "Replicate", "condition"],
    how="left",
)

# ------------------------------------------------------------
# Add phase-specific mean bout duration
# ------------------------------------------------------------

phase_bout_duration_wide = (
    df_bout_phase
    .pivot(
        index=["ID", "Replicate", "condition"],
        columns="Phase",
        values="mean_sleep_bout_duration_min",
    )
    .rename(
        columns={
            "Light": "mean_bout_duration_light_min",
            "Dark": "mean_bout_duration_dark_min",
        }
    )
    .reset_index()
)

df_sleep_parameters_R = df_sleep_parameters_R.merge(
    phase_bout_duration_wide,
    on=["ID", "Replicate", "condition"],
    how="left",
)

# ------------------------------------------------------------
# Add phase-specific number of bouts
# ------------------------------------------------------------

phase_bout_number_wide = (
    df_bout_phase
    .pivot(
        index=["ID", "Replicate", "condition"],
        columns="Phase",
        values="sleep_bout_number",
    )
    .rename(
        columns={
            "Light": "sleep_bout_number_light",
            "Dark": "sleep_bout_number_dark",
        }
    )
    .reset_index()
)

df_sleep_parameters_R = df_sleep_parameters_R.merge(
    phase_bout_number_wide,
    on=["ID", "Replicate", "condition"],
    how="left",
)

# Add metadata
df_sleep_parameters_R.insert(
    0,
    "Fragrance",
    FRAGRANCE_NAME,
)
df_sleep_parameters_R.insert(
    1,
    "Concentration",
    CONCENTRATION_LABEL,
)

# Make text columns export-safe
for col in ["Fragrance", "Concentration", "ID", "Replicate"]:
    df_sleep_parameters_R[col] = (
        df_sleep_parameters_R[col]
        .astype("string")
        .str.strip()
    )

df_sleep_parameters_R["condition"] = pd.Categorical(
    df_sleep_parameters_R["condition"],
    categories=CONDITION_ORDER,
    ordered=True,
)

df_sleep_parameters_R = (
    df_sleep_parameters_R
    .sort_values(["Replicate", "ID", "condition"])
    .reset_index(drop=True)
)

# Duplicate check: there should be exactly one row per ID × condition.
duplicates = df_sleep_parameters_R.duplicated(
    subset=["ID", "condition"],
    keep=False,
)

if duplicates.any():
    print("WARNING: duplicated ID × condition rows:")
    display(df_sleep_parameters_R.loc[duplicates])

print("\nCombined R dataframe:")
display(df_sleep_parameters_R.head())

print("\nShape:", df_sleep_parameters_R.shape)
print("Flies:", df_sleep_parameters_R["ID"].nunique())
print("\nColumns:")
print(df_sleep_parameters_R.columns.tolist())

In [ ]:
# ============================================================
# 11) EXPORT FOR R
# ============================================================

# Main endpoint dataframe:
# one row per fly × condition, all main sleep parameters in columns
R_PARAMETERS_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_sleep_parameters_for_R.csv"
)

df_sleep_parameters_R.to_csv(
    R_PARAMETERS_FILE,
    index=False,
)

# Hourly repeated-measures dataframe:
# use this for models with Hour and for the hourly sleep profile
df_hourly_sleep_R = df_hourly_sleep.copy()
df_hourly_sleep_R.insert(0, "Fragrance", FRAGRANCE_NAME)
df_hourly_sleep_R.insert(1, "Concentration", CONCENTRATION_LABEL)

R_HOURLY_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_hourly_sleep_for_R.csv"
)

df_hourly_sleep_R.to_csv(
    R_HOURLY_FILE,
    index=False,
)

# Optional audit data: one row per individual sleep bout
R_BOUTS_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_individual_sleep_bouts.csv"
)

sleep_bouts_overall.to_csv(
    R_BOUTS_FILE,
    index=False,
)

print("\nExports completed:")
print(f"1. Main sleep parameters: {R_PARAMETERS_FILE.resolve()}")
print(f"2. Hourly sleep data:     {R_HOURLY_FILE.resolve()}")
print(f"3. Individual bouts:      {R_BOUTS_FILE.resolve()}")

In [ ]:
# ============================================================
# 12) QUICK EXPORT CHECK
# ============================================================

expected_conditions = {"before", "after"}
found_conditions = set(
    df_sleep_parameters_R["condition"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nExport checks")
print("-------------")
print("Expected conditions present:",
      expected_conditions.issubset(found_conditions))

obs_per_fly = (
    df_sleep_parameters_R
    .groupby("ID", observed=True)
    .size()
)

print("Observations per fly:")
print(obs_per_fly.value_counts().sort_index())

incomplete_flies = obs_per_fly[obs_per_fly != 2]
print("Flies without exactly 2 condition rows:", len(incomplete_flies))

if len(incomplete_flies) > 0:
    display(
        incomplete_flies
        .rename("n_observations")
        .reset_index()
    )

print("\nDone.")

In [ ]:
# ============================================================
# EXPORT TOTAL SLEEP BY LIGHT/DARK FOR R
# ============================================================
#
# One row = one fly × condition × Phase
#
# Intended for statistical analysis analogous to total activity:
#
#   before vs after within Light
#   before vs after within Dark
#
# ============================================================

df_total_sleep_phase_R = (
    df_total_sleep_phase[
        [
            "ID",
            "Replicate",
            "condition",
            "Phase",
            "total_sleep_min",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Add metadata
# ------------------------------------------------------------

df_total_sleep_phase_R.insert(
    0,
    "Fragrance",
    FRAGRANCE_NAME,
)

df_total_sleep_phase_R.insert(
    1,
    "Concentration",
    CONCENTRATION_LABEL,
)


# ------------------------------------------------------------
# Clean text columns
# ------------------------------------------------------------

for col in [
    "Fragrance",
    "Concentration",
    "ID",
    "Replicate",
]:
    df_total_sleep_phase_R[col] = (
        df_total_sleep_phase_R[col]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# Keep factor order
# ------------------------------------------------------------

df_total_sleep_phase_R["condition"] = pd.Categorical(
    df_total_sleep_phase_R["condition"],
    categories=CONDITION_ORDER,
    ordered=True,
)

df_total_sleep_phase_R["Phase"] = pd.Categorical(
    df_total_sleep_phase_R["Phase"],
    categories=PHASE_ORDER,
    ordered=True,
)


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

df_total_sleep_phase_R = (
    df_total_sleep_phase_R
    .sort_values(
        [
            "Replicate",
            "ID",
            "condition",
            "Phase",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

R_TOTAL_SLEEP_PHASE_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_total_sleep_by_phase_for_R.csv"
)

df_total_sleep_phase_R.to_csv(
    R_TOTAL_SLEEP_PHASE_FILE,
    index=False,
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("\nTotal sleep by phase dataframe for R:")
display(df_total_sleep_phase_R.head(8))

print("\nShape:", df_total_sleep_phase_R.shape)
print("Number of flies:", df_total_sleep_phase_R["ID"].nunique())

print("\nObservations per fly:")
print(
    df_total_sleep_phase_R
    .groupby("ID", observed=True)
    .size()
    .value_counts()
    .sort_index()
)

print(
    f"\nSaved: {R_TOTAL_SLEEP_PHASE_FILE.resolve()}"
)